In [0]:
# Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Schema
schema = StructType([
    StructField('id', IntegerType()),
    StructField('visit_date', StringType()),
    StructField('people', IntegerType())
])

In [0]:
# DataFrame Example 1
data = [
    (1, '2017-01-01', 10),
    (2, '2017-01-02', 109),
    (3, '2017-01-03', 150),
    (4, '2017-01-04', 99),
    (5, '2017-01-05', 145),
    (6, '2017-01-06', 1455),
    (7, '2017-01-07', 199),
    (8, '2017-01-09', 188),
]

stadiumDf = spark.createDataFrame(data, schema)
stadiumDf.show()

In [0]:
def humanTraffic(df):
    # Converting Visit Date column to date type
    VisitDateCastDf = df.withColumn("visit_date", col("visit_date").cast("date"))

    # Filtering days when people >= 100
    FilteredDf = VisitDateCastDf.filter(col("people") >= 100)

    # Defining windows Spec
    windowSpec = Window.orderBy(col("id").asc())

    # Creating row number column and subtracting it from id as consecutive row will have same diff
    ConsecutiveDf = (
        FilteredDf
        .withColumn(
            "rowNum", 
            row_number().over(windowSpec)
        )
        .withColumn(
            "consecutiveRowNum", 
            (col("rowNum") - col("id"))
        )
    )

    # Counting number of rows for same group to find days where people >= 100 for at least 3 days
    GroupedDf = (
        ConsecutiveDf
        .groupBy(
            col("consecutiveRowNum")
        )
        .agg(
            count("*").alias("total")
        )
    )

    # Joining GroupedDf with ConsecutiveDf to find those days where people >= 100 for at least 3 days
    JoinedDf = (
        ConsecutiveDf
        .join(GroupedDf, on="consecutiveRowNum", how="inner")
        .filter(col("total") >= 3)
        .select(col("id"), col("visit_date"), col("people"))
    )

    return JoinedDf

In [0]:
humanTraffic(stadiumDf).show()